# Feature Engineering: Patient Readmission

**Purpose:** Document every feature engineering decision and validate that `src/features.py` produces the expected output.  
**Rule:** No production logic lives here. This notebook imports from `src/features.py` and tests it.

---

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.features import (
    engineer_features,
    build_target,
    deduplicate_patients,
    build_preprocessor,
    get_feature_names,
    map_icd9_to_group,
    NUMERIC_FEATURES,
    ORDINAL_FEATURES,
    NOMINAL_FEATURES,
)

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Imports OK')

## 1. Load Raw Data

In [ ]:
raw = pd.read_csv('../data/raw/diabetic_data.csv', na_values='?')
print(f'Raw shape: {raw.shape}')
print(f'Columns: {list(raw.columns)}')

## 2. Build Target Variable

In [ ]:
y_raw = build_target(raw)

print('Target variable distribution:')
print(f'  Positive (readmitted <30d): {y_raw.sum():,} ({y_raw.mean()*100:.1f}%)')
print(f'  Negative (not readmitted):  {(1-y_raw).sum():,} ({(1-y_raw).mean()*100:.1f}%)')
print()
print('Clinical rationale:')
print('  30-day threshold = CMS Hospital Readmissions Reduction Program standard')
print('  >30 and NO both map to 0 — we only penalize early readmissions')

## 3. Deduplication

In [ ]:
deduped = deduplicate_patients(raw)

print(f'Before deduplication: {len(raw):,} rows')
print(f'After deduplication:  {len(deduped):,} rows')
print(f'Rows removed:         {len(raw) - len(deduped):,}')
print()
print('Rationale: Multiple encounters from same patient violate statistical independence.')
print('Keeping first encounter = primary admission, consistent with Strack et al. (2014).')

## 4. ICD-9 Diagnosis Grouping

**Problem:** `diag_1` has 848 unique values. Using raw ICD-9 codes would:
- Create 848 one-hot columns → curse of dimensionality  
- Give the model no way to generalize across related conditions  

**Solution:** Map to 9 clinical groups following Strack et al. (2014).

In [ ]:
# Test the mapping function
test_codes = {
    '250.83': 'Diabetes',
    '410':    'Circulatory',
    '486':    'Respiratory',
    '820':    'Injury',
    'V27':    'Other',
    'E11':    'Other',
    None:     'Other',
}

print('ICD-9 mapping validation:')
all_pass = True
for code, expected in test_codes.items():
    result = map_icd9_to_group(code)
    status = '✓' if result == expected else '✗ FAIL'
    if result != expected:
        all_pass = False
    print(f'  {str(code):10s} → {result:20s} {status}')

print()
print(f'All tests passed: {all_pass}')

In [ ]:
# Show distribution reduction
print(f'diag_1 unique values before grouping: {raw["diag_1"].nunique()}')
grouped = raw['diag_1'].apply(map_icd9_to_group)
print(f'diag_1 unique values after grouping:  {grouped.nunique()}')
print()
print('Group distribution:')
print(grouped.value_counts().to_string())

## 5. Medication Feature Engineering

**Problem:** 24 medication columns, most with very low prescription rates.  
**Solution:** Two engineered features:
1. `on_insulin` — binary flag (insulin is prescribed by ~50% of encounters, strongest single drug signal)
2. `num_diabetes_meds` — count of all diabetes medications prescribed (captures medication burden)

In [ ]:
# Apply full feature engineering
y = build_target(deduped)
df_eng = engineer_features(deduped)

print(f'Shape after engineering: {df_eng.shape}')
print(f'Columns: {list(df_eng.columns)}')

In [ ]:
# Validate engineered features
print('on_insulin distribution:')
print(df_eng['on_insulin'].value_counts())
print(f'  → {df_eng["on_insulin"].mean()*100:.1f}% of patients on insulin')
print()

print('num_diabetes_meds distribution:')
print(df_eng['num_diabetes_meds'].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# on_insulin vs readmission
insulin_readmit = df_eng.copy()
insulin_readmit['readmitted_30d'] = y.values
rates = insulin_readmit.groupby('on_insulin')['readmitted_30d'].mean() * 100
axes[0].bar(['Not on Insulin', 'On Insulin'], rates.values, 
            color=['#3498db', '#e74c3c'])
axes[0].set_ylabel('30-day Readmission Rate (%)')
axes[0].set_title('Readmission Rate: Insulin vs No Insulin', fontweight='bold')
axes[0].axhline(y=y.mean()*100, color='black', linestyle='--', alpha=0.5, label='Overall avg')
axes[0].legend()

# num_diabetes_meds distribution
axes[1].hist(df_eng['num_diabetes_meds'], bins=20, color='#9b59b6', edgecolor='white')
axes[1].set_xlabel('Number of Diabetes Medications')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of num_diabetes_meds', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Build & Validate sklearn Pipeline

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df_eng, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {X_train.shape[0]:,} rows')
print(f'Test size:  {X_test.shape[0]:,} rows')
print(f'Train positive rate: {y_train.mean()*100:.1f}%')
print(f'Test positive rate:  {y_test.mean()*100:.1f}%')
print()
print('Stratified split preserves class imbalance in both sets ✓')

In [ ]:
# Fit the preprocessor
preprocessor = build_preprocessor()
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

feature_names = get_feature_names(preprocessor)

print(f'Input shape:   {X_train.shape}')
print(f'Output shape:  {X_train_processed.shape}')
print(f'Feature count: {len(feature_names)}')
print(f'\nFirst 10 feature names:')
for name in feature_names[:10]:
    print(f'  {name}')
print(f'  ... ({len(feature_names) - 10} more)')

In [ ]:
# Validate no NaNs in output
nan_count = np.isnan(X_train_processed).sum()
print(f'NaN values in processed training data: {nan_count}')
assert nan_count == 0, 'Pipeline produced NaN values — check imputers'

nan_count_test = np.isnan(X_test_processed).sum()
print(f'NaN values in processed test data:     {nan_count_test}')
assert nan_count_test == 0, 'Pipeline produced NaN values in test set'

print()
print('Pipeline validation passed ✓')
print('  - No NaN values in output')
print(f'  - Output shape: {X_train_processed.shape}')
print(f'  - All {len(feature_names)} features named and retrievable')

## 7. Feature Engineering Summary

| Decision | Rationale |
|---|---|
| Drop `weight` | 96.9% missing — imputation not viable |
| Drop `payer_code` | Not clinically relevant to readmission |
| Keep `medical_specialty`, impute 'Unknown' | Admitting physician type is clinically meaningful |
| Keep `race`, impute 'Unknown' | Required for fairness audit |
| Deduplicate to first encounter per patient | Preserves statistical independence |
| Map ICD-9 → 9 clinical groups | Reduces 848 unique codes to interpretable groups |
| `on_insulin` binary flag | Strongest single medication signal |
| `num_diabetes_meds` count | Captures medication burden without 24 sparse columns |
| Ordinal encode `age`, `A1Cresult`, `max_glu_serum` | Preserves clinical ordering |
| One-hot encode nominal features | No ordinal assumption imposed |
| StandardScaler on numerics | Required for Logistic Regression convergence |